In [ ]:
# install requirements
!pip install -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd
import scikit as sklearn
from scipy import signal

# Feature Extraction

In [ ]:
ACCEL_AXES = [0, 1, 2]       # ax, ay, az
GYRO_AXES  = [3, 4, 5]       # gx, gy, gz
# if virtual streams added:   index 6 = accel_mag, index 7 = gyro_mag

## Time Domain

- features based off of the time domain (samples per unit time)
- Time-domain features are cheap, interpretable, and surprisingly powerful for punch classification.
- Mean, std, RMS, peak are the baseline — start here.
- Add skewness/kurtosis if classes are hard to separate with basics.
- Add ZCR and MAD for extra discriminative power at low cost.

In [ ]:
# INPUT: 2D (samples, channels) --> takes in one window
# OUTPUT: 1D (features) --> does statistical analysis per axes to result in channels * number of features


## output: 8 features per channel --> channels * 8
def extract_time_domain(window):
    """
    Extracts time-domain statistics for each axis in a single window.
 
    Input:  (window_size, n_axes)   e.g. (100, 8)
    Output: 1D feature vector,      e.g. (n_axes * 8,) = (64,) for 8 axes
 
    Features per axis (8 total):
        mean     — average value. Captures DC offset / resting orientation.
        std      — spread of values. High during punch, low at rest.
        rms      — root mean square. Energy of the signal. Good punch detector.
        max      — peak positive value.
        min      — peak negative value.
        peak     — max(abs). Overall peak regardless of direction.
        skewness — asymmetry of distribution. Differs between punch types.
        kurtosis — peakedness. High for sharp impact spikes.
 
    WHY these specifically:
        mean + std   → captures where the signal lives and how active it is
        rms          → better than mean for oscillating signals (punches)
        peak         → captures impact intensity
        skew + kurt  → captures the shape of the motion, differs per punch type
    """
    features = []
    for axis in range(window.shape[1]):
        x = window[:, axis]
        features.extend([
            np.mean(x),
            np.std(x),
            np.sqrt(np.mean(x ** 2)),           # RMS
            np.max(x),
            np.min(x),
            np.max(np.abs(x)),                  # peak
            float(skew(x)),                     # asymmetry
            float(kurtosis(x)),                 # peakedness
        ])
    return np.array(features, dtype=np.float32)


# output: 1 feature (zcr) per channel
def extract_zcr(window):
    """
    Zero Crossing Rate (ZCR) per axis — how many times the signal crosses zero.
 
    Input:  (window_size, n_axes)
    Output: (n_axes,)
 
    GOOD FOR:   distinguishing punches from rest. A punch has high ZCR
                (oscillating motion), rest has near-zero ZCR.
                Also differs between punch types — uppercut has fewer
                crossings (one clean arc) vs hook (more oscillation).
    CHEAP:      one number per axis, very fast to compute.
    """
    signs = np.sign(window)
    # count sign changes along time axis
    crossings = np.sum(np.abs(np.diff(signs, axis=0)) > 0, axis=0)
    return crossings.astype(np.float32)


# output: 1 feature per channel
def extract_mad(window):
    """
    Mean Absolute Deviation per axis.
 
    Input:  (window_size, n_axes)
    Output: (n_axes,)
 
    GOOD FOR:   robust alternative to std. Less sensitive to outliers/spikes.
                Use alongside std — if they differ a lot, there are spikes.
    """
    return np.mean(np.abs(window - np.mean(window, axis=0)), axis=0).astype(np.float32)


# output : 1 feature per ACCELEROMETER channel (3,) # CAUTION AS LENGTH MAY DIFFER WITH OTHER MATRICES SINCE IT IS ONLY COMPUTED ON THE 3 ACCELEROMETER AXES ONLY
def extract_jerk(window, fs=100):
    """
    Jerk = derivative of acceleration (rate of change).
    Computed as mean absolute jerk per accel axis.
 
    Input:  (window_size, n_axes)
    Output: (3,)  — one value per accel axis
 
    GOOD FOR:   capturing the sharpness of the punch onset.
                A straight punch has a sharp, clean jerk spike.
                An uppercut has a more gradual jerk profile.
                Very discriminative for punch type classification.
    NOTE:       applies to accel axes only [0,1,2].
                Use Savitzky-Golay filtered data for cleanest derivative.
    """
    accel = window[:, ACCEL_AXES]                        # (window_size, 3)
    jerk = np.diff(accel, axis=0) * fs                   # (window_size-1, 3)
    return np.mean(np.abs(jerk), axis=0).astype(np.float32)


## Frequency Domain

- features based off of the frequency values

- When time-domain features alone aren't enough to separate classes.
- Punches have characteristic frequency signatures:
   straight  → clean dominant frequency, fast onset
     hook      → broader spectrum, more rotation content
     uppercut  → lower dominant frequency, more vertical content
  - Adds meaningful features at moderate compute cost.
  - SKIP if your window is very short (<0.5s) — FFT resolution suffers.

In [ ]:
#INPUT: single window --> (window_len, channels)
#OUTPUT: number of features extracted * channels

# output: channels * 4 (extracts 4 key fft features)
def extract_fft_features(window, fs=100):
    """
    FFT-based frequency features per axis.
 
    Input:  (window_size, n_axes)
    Output: 1D feature vector, (n_axes * 4,)
 
    Features per axis (4 total):
        dominant_freq  — frequency with highest magnitude. The main
                         oscillation frequency of the punch.
        spectral_energy— total energy in the frequency domain. Correlates
                         with punch intensity.
        spectral_mean  — mean of the magnitude spectrum (weighted avg freq).
        spectral_std   — spread of energy across frequencies.
 
    NOTE: FFT resolution = fs / window_size.
          At 100 Hz, 100 samples → 1 Hz resolution. Good.
          At 100 Hz, 50 samples  → 2 Hz resolution. Still ok.
    """
    features = []
    n = window.shape[0]
    freqs = np.fft.rfftfreq(n, d=1.0 / fs)              # frequency bins
 
    for axis in range(window.shape[1]):
        x = window[:, axis]
        mag = np.abs(np.fft.rfft(x))                    # magnitude spectrum
 
        dominant_freq   = freqs[np.argmax(mag)]
        spectral_energy = np.sum(mag ** 2) / n
        spectral_mean   = np.sum(freqs * mag) / (np.sum(mag) + 1e-9)
        spectral_std    = np.sqrt(
            np.sum(((freqs - spectral_mean) ** 2) * mag) / (np.sum(mag) + 1e-9)
        )
 
        features.extend([dominant_freq, spectral_energy, spectral_mean, spectral_std])
 
    return np.array(features, dtype=np.float32)


# output: channels * 4 (4 freq bands extracted)
def extract_band_energy(window, fs=100, bands=None):
    """
    Energy in specific frequency bands per axis.
 
    Input:  (window_size, n_axes)
    Output: (n_axes * n_bands,)
 
    Default bands are tuned for punch data:
        (0,  2)  Hz  — drift / very slow motion     (should be near zero)
        (2,  10) Hz  — main punch motion band
        (10, 20) Hz  — wrist snap / fast motion
        (20, 50) Hz  — noise band                   (should be low after LP filter)
 
    GOOD FOR:   understanding WHERE in the spectrum the energy lives.
                A straight punch concentrates energy in 2-10 Hz.
                A hook spreads energy more into 10-20 Hz (wrist rotation).
    """
    if bands is None:
        bands = [(0, 2), (2, 10), (10, 20), (20, 50)]
 
    features = []
    n = window.shape[0]
    freqs = np.fft.rfftfreq(n, d=1.0 / fs)
 
    for axis in range(window.shape[1]):
        x = window[:, axis]
        mag = np.abs(np.fft.rfft(x)) ** 2              # power spectrum
 
        for low, high in bands:
            band_mask = (freqs >= low) & (freqs < high)
            energy = np.sum(mag[band_mask]) / (n + 1e-9)
            features.append(energy)
 
    return np.array(features, dtype=np.float32)
 


#stft --> extract features based on how frequency changes over time. Only effective on windows of at least >0.5s+
# - Useful for capturing punch onset vs follow-through — the beginning
#     of a punch looks different from the end in frequency space.
#   - More expensive than FFT. Only add if FFT features aren't enough.
# output: channels * 4 (extract features from the STFT)
def extract_stft_features(window, fs=100, nperseg=32):
    """
    Summarizes the STFT spectrogram into a flat feature vector.
 
    Input:  (window_size, n_axes)
    Output: (n_axes * 4,)  — mean/std/max/energy of the spectrogram per axis
 
    nperseg=32  →  each sub-window is 32 samples = 320ms @ 100Hz.
                   smaller = more time resolution, less freq resolution.
                   larger  = more freq resolution, less time resolution.
 
    GOOD FOR:   capturing temporal evolution of frequency content.
                e.g. "energy starts low then spikes" = punch onset pattern.
    NOTE:       returns summary stats of the spectrogram, NOT the full
                spectrogram (which would be too large for SVM/RF input).
    """
    features = []
    for axis in range(window.shape[1]):
        x = window[:, axis]
        _, _, Zxx = signal.stft(x, fs=fs, nperseg=nperseg)
        mag = np.abs(Zxx)                               # (freq_bins, time_frames)
 
        features.extend([
            np.mean(mag),                               # average spectral content
            np.std(mag),                                # variation over time
            np.max(mag),                                # peak spectral energy
            np.sum(mag ** 2) / mag.size,                # mean power
        ])
 
    return np.array(features, dtype=np.float32)



## Correlation Features

 - Captures relationships BETWEEN axes, not just per-axis stats.
 - A straight punch: ax and ay move somewhat independently.
   A hook: ax and gy (rotation) are highly correlated.
 - Cheap to compute, adds cross-axis information that per-axis
  features completely miss.

In [ ]:

#INPUT: single window --> (window_len, channels)
#OUTPUT: (channels * num_features )

#output: (channels * (channels-1)/2 ) # CAUTION: USE WITH CAUTION SINCE THE MATRIX DIMENSIONS MAY DIFFER
def extract_correlations(window):
    """
    Pairwise Pearson correlations between axes.
 
    Input:  (window_size, n_axes)
    Output: (n_axes*(n_axes-1)/2,)  — upper triangle of correlation matrix
            e.g. 6 axes → 15 values, 8 axes → 28 values
 
    WHY upper triangle only:
        corr(ax, ay) == corr(ay, ax), and corr(ax, ax) == 1 always.
        No point including duplicates or self-correlations.
    """
    n_axes = window.shape[1]
    corr = np.corrcoef(window.T)                        # (n_axes, n_axes)
    # extract upper triangle (excluding diagonal)
    idx = np.triu_indices(n_axes, k=1)
    return corr[idx].astype(np.float32)


## PIPELINE
- for each window, pass it to the relevant feature exrtaction methods to extract as many features that can help encode relevant info about that window of time.

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5. MASTER FEATURE EXTRACTOR
# ─────────────────────────────────────────────────────────────
 
def extract_features(windows, fs=100, use_fft=True, use_stft=False,
                     use_band_energy=True, use_correlations=True,
                     use_jerk=True, use_zcr=True, use_mad=True):
    """
    Runs all selected feature extractors over every window.
 
    Input:  windows — (n_windows, window_size, n_axes)
    Output: X       — (n_windows, n_features)
 
    Default feature set and rough feature count for 8 axes:
        time_domain    →  8 axes * 8 stats  =  64
        zcr            →  8 axes            =   8
        mad            →  8 axes            =   8
        jerk           →  3 (accel only)    =   3
        fft            →  8 axes * 4 stats  =  32
        band_energy    →  8 axes * 4 bands  =  32
        correlations   →  8*(8-1)/2         =  28
                                         ───────
                                         ~175 features total
 
    Toggle individual groups on/off with the boolean flags.
    Start with use_stft=False — only enable if accuracy plateaus.
    """
    all_features = []
 
    for i in range(windows.shape[0]):
        w = windows[i]                                  # (window_size, n_axes)
        feats = []
 
        # always include time domain
        feats.append(extract_time_domain(w))
 
        if use_zcr:
            feats.append(extract_zcr(w))
 
        if use_mad:
            feats.append(extract_mad(w))
 
        if use_jerk:
            feats.append(extract_jerk(w, fs=fs))
 
        if use_fft:
            feats.append(extract_fft_features(w, fs=fs))
 
        if use_band_energy:
            feats.append(extract_band_energy(w, fs=fs))
 
        if use_stft:
            feats.append(extract_stft_features(w, fs=fs))
 
        if use_correlations:
            feats.append(extract_correlations(w))
 
        all_features.append(np.concatenate(feats))
 
    X = np.stack(all_features, axis=0)                 # (n_windows, n_features)
    return X.astype(np.float32)
 
 